# Phase 2 — Preprocessing

**Project:** GradeLens — Habit-Based Academic Performance Analysis
**Notebook:** `02_preprocessing.ipynb`
**Purpose:** Clean the raw data, handle missing values, drop redundant columns, scale numeric features, and produce the final canonical datasets for EDA and K-Means clustering.

---


In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
import os

RAW_PATH = "../data/raw/college_students_habits_1M.csv"
OUT_CLEAN = "../data/processed/students_clean.csv"
OUT_SCALED = "../data/processed/students_scaled.csv"

df = pd.read_csv(RAW_PATH)
print("Initial shape:", df.shape)

Initial shape: (1000000, 42)


## 1. Feature Review & Dropping Columns

Based on the Phase 1 schema contract, we must drop the following columns:
- `performance_level`: Contains only 'Low' (zero variance).
- `screen_time`: Derived column (sum of other screen activities).
- `self_discipline`: Perfectly correlated with `time_management`.
- `netflix_hours` & `late_night_frequency`: Perfectly correlated with `gaming_hours`.

In [2]:
cols_to_drop = [
    "performance_level",
    "screen_time",
    "self_discipline",
    "netflix_hours",
    "late_night_frequency"
]

df.drop(columns=[c for c in cols_to_drop if c in df.columns], inplace=True)
print("Shape after dropping columns:", df.shape)

Shape after dropping columns: (1000000, 37)


## 2. Duplicate Rows & Near-Zero Variance

We will check for and drop exact duplicate rows. We also check for columns with near-zero variance.

In [3]:
# Drop duplicates
initial_rows = len(df)
df.drop_duplicates(inplace=True)
print(f"Dropped {initial_rows - len(df)} duplicate rows.")

# Check for near-zero variance
numeric_cols = df.select_dtypes(include=[np.number]).columns
variances = df[numeric_cols].var()
near_zero = variances[variances < 0.01]
print("\nColumns with near-zero variance (< 0.01):")
print(near_zero if not near_zero.empty else "None found.")

Dropped 0 duplicate rows.

Columns with near-zero variance (< 0.01):
None found.


## 3. Missing Values Strategy

We will:
1. Drop any row where the primary target (`gpa`) is missing.
2. Impute continuous numeric columns with the **median** (robust to skew).
3. Impute categorical/ordinal columns with the **mode**.

In [4]:
print("Missing values before imputation:")
missing = df.isnull().sum()
print(missing[missing > 0])

# 1. Drop if target is missing
df.dropna(subset=['gpa'], inplace=True)

# 2. Identify categoricals/ordinals based on Phase 1 spec
categorical_cols = ['parental_education_level', 'peer_study_group', 'relationship_status', 'hostel_student']

# 3. Impute
for col in df.columns:
    if df[col].isnull().any():
        if col in categorical_cols:
            mode_val = df[col].mode()[0]
            df[col].fillna(mode_val, inplace=True)
        else:
            median_val = df[col].median()
            df[col].fillna(median_val, inplace=True)

print("\nMissing values after imputation:", df.isnull().sum().sum())

Missing values before imputation:
Series([], dtype: int64)

Missing values after imputation: 0


## 4. Normalization and Categorical Encoding

From Phase 1:
- `previous_gpa` is on a 0-10 scale, but `gpa` is on a 0-2 scale. We will divide `previous_gpa` by 5 to normalize it.
- The nominal and ordinal categorical columns are already encoded as integers (`0/1` or `1-5`). Thus, no additional one-hot encoding is necessary.

In [5]:
if df['previous_gpa'].max() > 2.0:
    df['previous_gpa'] = df['previous_gpa'] / 5.0
    print("Normalized `previous_gpa` to 0-2 scale.")

# Ensure categoricals are integers
for col in categorical_cols:
    if col in df.columns:
        df[col] = df[col].astype('int8')

Normalized `previous_gpa` to 0-2 scale.


## 5. Scaling (For Clustering)

We need to create a scaled dataset for Person A's K-Means clustering. We will apply `StandardScaler` (zero mean, unit variance) to all `numeric_feature` columns. The target (`gpa`) and `categorical_feature` columns are not scaled.

We will export two dataframes:
1. `students_clean.csv`: Clean, but **unscaled** (used for EDA and Association Rules).
2. `students_scaled.csv`: Clean, and **scaled** (used for Clustering).

In [6]:
df_clean = df.copy()

# Numeric features for clustering (excluding target and categoricals)
exclude_from_scaling = categorical_cols + ['gpa', 'midterm_score', 'final_score', 'project_score']
numeric_features = [c for c in df.columns if c not in exclude_from_scaling]

scaler = StandardScaler()
df_scaled = df.copy()
df_scaled[numeric_features] = scaler.fit_transform(df_scaled[numeric_features])

print("Scaling complete. Example scaled column (study_hours):")
print(df_scaled['study_hours'].head())

Scaling complete. Example scaled column (study_hours):
0   -0.464290
1   -0.170268
2   -0.604795
3   -0.269785
4   -1.739632
Name: study_hours, dtype: float64


## 6. Export Datasets

In [7]:
df_clean.to_csv(OUT_CLEAN, index=False)
df_scaled.to_csv(OUT_SCALED, index=False)

print(f"Successfully exported:\n - {OUT_CLEAN} ({df_clean.shape})\n - {OUT_SCALED} ({df_scaled.shape})")

Successfully exported:
 - ../data/processed/students_clean.csv ((1000000, 37))
 - ../data/processed/students_scaled.csv ((1000000, 37))
